# CDR-MLC — leakage-safe Table 5 reproduction

This notebook is only an experiment interface. The authoritative model is `cdr_mlc/model.py`, and the authoritative scenario runner is `run_table5.py`. No training logic is duplicated here.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'run_table5.py').exists():
    raise RuntimeError('Start Jupyter from the CDR_MLC directory.')
DATA_ROOT = PROJECT_ROOT / 'DATASETS/CDR-MLC/scale_0.001'
OUTPUT_DIR = PROJECT_ROOT / 'results/runs/table5'
SEEDS = [42]  # final repeated run: [10, 20, 30, 40, 42]
DATA_ROOT, OUTPUT_DIR

In [ ]:
from run_table5 import scenario_paths

paths = scenario_paths(DATA_ROOT)
missing = sorted({str(path) for pair in paths.values() for path in pair if not path.exists()})
if missing:
    raise FileNotFoundError('Missing required dataset files:\n' + '\n'.join(missing))
pd.DataFrame([
    {'scenario': name, 'train': str(train), 'test': str(test)}
    for name, (train, test) in paths.items()
])

## Run experiments
The command runs scenarios and seeds sequentially to control peak memory. Start with seed 42.

In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / 'run_table5.py'),
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--seeds', *map(str, SEEDS),
]
subprocess.run(command, check=True)

In [ ]:
runs = pd.read_csv(OUTPUT_DIR / 'runs.csv')
metric_columns = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted', 'f1_macro']
display_table = runs[['scenario', 'seed', *metric_columns]].copy()
display_table[metric_columns] = (100 * display_table[metric_columns]).round(2)
display_table

In [ ]:
summary = runs.groupby('scenario')[metric_columns].agg(['mean', 'std'])
summary = (100 * summary).round(2)
summary

In [ ]:
plot_data = runs.groupby('scenario')[['accuracy', 'f1_weighted', 'f1_macro']].mean() * 100
ax = plot_data.plot.bar(figsize=(10, 5), ylim=(0, 100), rot=0)
ax.set_title('Leakage-safe CDR-MLC — Table 5 scenarios')
ax.set_xlabel('Scenario')
ax.set_ylabel('Score (%)')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()